In [1]:
import sys
sys.path.append("../src")

In [2]:
import pandas as pd 
import numpy as np
from pathlib import Path
from modelado import fechas_estacion, calcular_grados, filtrar_entrenamiento, calcular_temperatura_media, armar_futuro, pronosticar_cuantiles, asignar_estaciones, calcular_maximos_estacionales
from regulacion import proyectar_cargo

In [3]:
ruta_serie = Path("../data/processed/SERIE_TRABAJO_CTRELEUW.csv")
serie = pd.read_csv(ruta_serie, parse_dates=["MES"])

In [4]:
inicio, fin = fechas_estacion("VERANO", 2026)

In [5]:
pasado = serie[serie["MES"] < inicio].copy()
pasado = calcular_grados(pasado)

In [6]:
entrenamiento = filtrar_entrenamiento(pasado)
temp_media = calcular_temperatura_media(pasado)
futuro = armar_futuro(inicio, fin, temp_media)

In [7]:
tabla_ordenada = pronosticar_cuantiles(entrenamiento, futuro)

In [8]:
futuro[["Q10_MW", "Q50_MW", "Q90_MW"]] = tabla_ordenada[[0.1, 0.5, 0.9]].values
futuro[["MES", "Q10_MW", "Q50_MW", "Q90_MW"]]

,MES,Q10_MW,Q50_MW,Q90_MW
0,2026-11-01,38.038141,40.831556,44.566223
1,2026-12-01,38.764093,42.091490,45.491887
2,2027-01-01,39.733881,43.525924,46.545755
3,2027-02-01,39.128857,42.724561,45.956999
4,2027-03-01,38.185075,41.086567,44.753578
5,2027-04-01,36.470809,38.106152,42.563889


In [9]:
serie = asignar_estaciones(serie)
maximos = calcular_maximos_estacionales(serie)

In [10]:
futuro = proyectar_cargo(futuro, maximos, "VERANO", 2026)
futuro[["MES", "Q10_MW", "Q50_MW", "Q90_MW", "PISO_MW", "HRP", "CARGO_Q10_USD", "CARGO_Q50_USD", "CARGO_Q90_USD"]]

,MES,Q10_MW,Q50_MW,Q90_MW,PISO_MW,HRP,CARGO_Q10_USD,CARGO_Q50_USD,CARGO_Q90_USD
0,2026-11-01,38.038141,40.831556,44.566223,39.82,378,216748.22,222254.33,242582.86
1,2026-12-01,38.764093,42.091490,45.491887,39.82,414,290144.45,306695.44,331472.08
2,2027-01-01,39.733881,43.525924,46.545755,39.82,378,264914.50,289569.27,309659.60
3,2027-02-01,39.128857,42.724561,45.956999,39.82,360,252299.52,270702.82,291183.55
4,2027-03-01,38.185075,41.086567,44.753578,39.82,414,237390.91,244941.68,266802.93
5,2027-04-01,36.470809,38.106152,42.563889,39.82,396,227069.57,227069.57,242716.32


In [11]:
cargo_estacion_cuantil = futuro[["CARGO_Q10_USD", "CARGO_Q50_USD", "CARGO_Q90_USD"]].sum()
cargo_estacion_cuantil

CARGO_Q10_USD    1488567.17
CARGO_Q50_USD    1561233.11
CARGO_Q90_USD    1684417.34
dtype: float64